In [1]:
print('hello')

hello


In [2]:
import requests
import pandas as pd
import re
from datetime import datetime

# -----------------------------
# RemoteOK API
# -----------------------------
API_URL = "https://remoteok.com/api"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# Fetch Data
# -----------------------------
response = requests.get(API_URL, headers=headers)

data = response.json()

# Skip metadata row
jobs = data[1:]

print(f"\nTotal Jobs Fetched: {len(jobs)}")

# -----------------------------
# Salary Regex Patterns
# -----------------------------
salary_patterns = [
    r'\$\d+[kK]?',
    r'\d+\s?USD',
    r'\€\d+[kK]?',
    r'\₹\d+[lL]?[pP]?[aA]?',
    r'\d+[kK]\s?-\s?\d+[kK]'
]

# Combine patterns
combined_pattern = '|'.join(salary_patterns)

filtered_jobs = []

# -----------------------------
# Process Jobs
# -----------------------------
for job in jobs:

    description = str(job.get("description", ""))
    salary_raw = str(job.get("salary", ""))

    # Search salary in salary field OR description
    salary_match = re.search(combined_pattern, salary_raw + " " + description)

    if salary_match:

        filtered_jobs.append({
            "job_id": job.get("id"),
            "position": job.get("position"),
            "company": job.get("company"),
            "location": job.get("location"),
            "salary_detected": salary_match.group(),
            "tags": ", ".join(job.get("tags", [])),
            "date_posted": job.get("date"),
            "url": job.get("url")
        })

# -----------------------------
# Create DataFrame
# -----------------------------
df = pd.DataFrame(filtered_jobs)

print(f"\nJobs With Salary Information: {len(df)}")

# -----------------------------
# Show Sample
# -----------------------------
print("\n===== FILTERED DATASET =====\n")
print(df.head(10))

# -----------------------------
# Save CSV
# -----------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_file = f"remoteok_salary_jobs_{timestamp}.csv"

df.to_csv(csv_file, index=False)

print(f"\nFiltered CSV Saved: {csv_file}")


Total Jobs Fetched: 93

Jobs With Salary Information: 23

===== FILTERED DATASET =====

    job_id                                           position  \
0  1131715                Asset Protection Specialist BURNABY   
1  1131773  Agente de Recrutamento e SeleÃ§Ã£o Remoto Diar...   
2  1131862                                        Proofreader   
3  1131696                                 Operations Analyst   
4  1131717                         Loss Prevention Specialist   
5  1131709                     Head of Operations Overtime.ai   
6  1131711                   Chief Medical Scribe Upto $80 hr   
7  1131702             Pediatric Registered Nurse Upto $65 hr   
8  1131904                                   Graphic Designer   
9  1131920           Investment Research Analyst Upto $100 hr   

                 company                                     location  \
0  The Home Depot Canada                                    Burnaby,    
1               Solvereh          Londrina, Londr

In [3]:
df2  = pd.read_csv('remoteok_salary_jobs_20260523_172321.csv')
df2

,job_id,position,company,location,salary_detected,tags,date_posted,url
0,1131715,Asset Protection Specialist BURNABY,The Home Depot Canada,"Burnaby,",$22,"travel, non tech, testing, test, support, recr...",2026-05-22T01:04:25+00:00,https://remoteOK.com/remote-jobs/remote-asset-...
1,1131773,Agente de Recrutamento e SeleÃ§Ã£o Remoto Diar...,Solvereh,"Londrina, Londrina, ParanÃ¡, Brasil",$80,"design, gaming, marketing, photoshop, branding...",2026-05-22T00:53:24+00:00,https://remoteOK.com/remote-jobs/remote-agente...
2,1131862,Proofreader,Mower Agency,"New York, New York, United States",$28,"assistant, support, testing, management, opera...",2026-05-22T00:00:00+00:00,https://remoteOK.com/remote-jobs/remote-proofr...
3,1131696,Operations Analyst,Tremendous,"New York, New York, New York, United States",$95k,"analyst, support, growth, director, manager, f...",2026-05-21T19:57:03+00:00,https://remoteOK.com/remote-jobs/remote-operat...
4,1131717,Loss Prevention Specialist,The Home Depot Canada,"Kitchener,",$22,"travel, non tech, testing, test, support, recr...",2026-05-21T19:03:23+00:00,https://remoteOK.com/remote-jobs/remote-loss-p...
5,1131709,Head of Operations Overtime.ai,Acclaim AI,"Miami, Miami, Florida, United States",$20,"analyst, support, growth, director, manager, f...",2026-05-21T18:43:26+00:00,https://remoteOK.com/remote-jobs/remote-head-o...
6,1131711,Chief Medical Scribe Upto $80 hr,Mercor,"New York, New York, New York, United States",$55,"analyst, support, growth, director, manager, f...",2026-05-21T02:22:15+00:00,https://remoteOK.com/remote-jobs/remote-chief-...
7,1131702,Pediatric Registered Nurse Upto $65 hr,Mercor,"New York, New York, New York, United States",$55,"analyst, support, growth, director, manager, f...",2026-05-21T02:19:20+00:00,https://remoteOK.com/remote-jobs/remote-pediat...
8,1131904,Graphic Designer,Mercor,"Sydney, Sydney, New South Wales, Australia",$60,"video, ads, content, marketing, illustrator, e...",2026-05-21T02:18:47+00:00,https://remoteOK.com/remote-jobs/remote-graphi...
9,1131920,Investment Research Analyst Upto $100 hr,Mercor,"London, London, England, United Kingdom",$100,"customer support, support, growth, financial, ...",2026-05-21T02:18:40+00:00,https://remoteOK.com/remote-jobs/remote-invest...


In [4]:
import requests
import pandas as pd
import re
from datetime import datetime

# =====================================================
# FUNCTION: Extract Salary Range
# =====================================================

def extract_salary_range(text):

    if not text:
        return None, None, None

    text = text.replace(",", "")

    patterns = [

        # $120k-$180k
        (r'\$?(\d+)[kK]\s*[-to]+\s*\$?(\d+)[kK]', 'USD', 1000),

        # €80k-€120k
        (r'€?(\d+)[kK]\s*[-to]+\s*€?(\d+)[kK]', 'EUR', 1000),

        # ₹20LPA-₹35LPA
        (r'₹?(\d+)[lL][pP][aA]\s*[-to]+\s*₹?(\d+)[lL][pP][aA]', 'INR', 100000),

        # 120000-180000 USD
        (r'(\d{2,6})\s*[-to]+\s*(\d{2,6})\s*USD', 'USD', 1),

        # 80000 to 120000
        (r'(\d{2,6})\s*(?:-|to)\s*(\d{2,6})', 'UNKNOWN', 1),
    ]

    for pattern, currency, multiplier in patterns:

        match = re.search(pattern, text, re.IGNORECASE)

        if match:

            min_salary = int(match.group(1)) * multiplier
            max_salary = int(match.group(2)) * multiplier

            return min_salary, max_salary, currency

    return None, None, None


# =====================================================
# REMOTEOK API
# =====================================================

API_URL = "https://remoteok.com/api"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# =====================================================
# FETCH DATA
# =====================================================

response = requests.get(API_URL, headers=headers)

response.raise_for_status()

data = response.json()

# First element is metadata
jobs = data[1:]

print(f"\nTotal Jobs Fetched: {len(jobs)}")

# =====================================================
# PROCESS JOBS
# =====================================================

filtered_jobs = []

for job in jobs:

    description = str(job.get("description", ""))
    salary_field = str(job.get("salary", ""))

    combined_text = salary_field + " " + description

    salary_min, salary_max, currency = extract_salary_range(combined_text)

    # Keep only jobs where salary was found
    if salary_min and salary_max:

        filtered_jobs.append({

            "job_id": job.get("id"),

            "position": job.get("position"),

            "company": job.get("company"),

            "location": job.get("location"),

            "salary_min": salary_min,

            "salary_max": salary_max,

            "currency": currency,

            "tags": ", ".join(job.get("tags", [])),

            "date_posted": job.get("date"),

            "url": job.get("url")
        })

# =====================================================
# CREATE DATAFRAME
# =====================================================

df = pd.DataFrame(filtered_jobs)

print(f"\nJobs With Salary Range Found: {len(df)}")

# =====================================================
# DISPLAY SAMPLE
# =====================================================

print("\n===== FILTERED DATASET =====\n")

print(df.head(20))

# =====================================================
# SAVE CSV
# =====================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_file = f"remoteok_salary_jobs_{timestamp}.csv"

df.to_csv(csv_file, index=False)

print(f"\nCSV Saved Successfully: {csv_file}")


Total Jobs Fetched: 93

Jobs With Salary Range Found: 3

===== FILTERED DATASET =====

    job_id                              position         company  \
0  1131696                    Operations Analyst      Tremendous   
1  1131704                 Special Projects Lead      Tremendous   
2  1131799  Customer Success Manager Great Lakes  PointClickCare   

                                      location  salary_min  salary_max  \
0  New York, New York, New York, United States       95000      105000   
1  New York, New York, New York, United States      170000      200000   
2    Mississauga, Mississauga, Ontario, Canada          25          50   

  currency                                               tags  \
0      USD  analyst, support, growth, director, manager, f...   
1      USD  analyst, support, growth, director, manager, f...   
2  UNKNOWN  training, recruiting, educational, executive, ...   

                 date_posted  \
0  2026-05-21T19:57:03+00:00   
1  2026-05-21T01:

In [5]:
df2  = pd.read_csv('remoteok_salary_jobs_20260523_172628.csv')
df2

,job_id,position,company,location,salary_min,salary_max,currency,tags,date_posted,url
0,1131696,Operations Analyst,Tremendous,"New York, New York, New York, United States",95000,105000,USD,"analyst, support, growth, director, manager, f...",2026-05-21T19:57:03+00:00,https://remoteOK.com/remote-jobs/remote-operat...
1,1131704,Special Projects Lead,Tremendous,"New York, New York, New York, United States",170000,200000,USD,"analyst, support, growth, director, manager, f...",2026-05-21T01:56:32+00:00,https://remoteOK.com/remote-jobs/remote-specia...
2,1131799,Customer Success Manager Great Lakes,PointClickCare,"Mississauga, Mississauga, Ontario, Canada",25,50,UNKNOWN,"training, recruiting, educational, executive, ...",2026-05-18T20:16:45+00:00,https://remoteOK.com/remote-jobs/remote-custom...
